# π₀.₅ (Pi0.5) — Vision-Language-Action Policy on AMD ROCm

**Platform:** AMD ROCm (gfx1100, 48 GB VRAM)  
**Model:** [`lerobot/pi05_base`](https://huggingface.co/lerobot/pi05_base)  
**Framework:** [LeRobot](https://github.com/huggingface/lerobot) by Hugging Face  
**Architecture:** Vision-Language-Action (VLA) — open-world generalization  
**License:** Apache 2.0

π₀.₅ is Physical Intelligence's open-world robot policy, adapted in LeRobot. It co-trains on heterogeneous data (web images, verbal instructions, cross-embodiment robot demos) to generalize across environments, objects, and tasks never seen during training.

---

## Hardware Requirements

| Component | Requirement | This machine |
|---|---|---|
| GPU VRAM | ~24 GB (BF16, `train_expert_only`) | ✅ 48 GB gfx1100 |
| GPU VRAM | ~40 GB (BF16, full finetune) | ✅ fits |
| BF16 support | Required | ✅ RDNA3 native |
| Python | ≥ 3.12 | check below |

## 0. Environment Verification

Verify the ROCm stack and Python version. On AMD hardware, PyTorch exposes HIP through the standard `torch.cuda` API.

In [1]:
import subprocess, sys, os, torch

print("=" * 55)
print("ROCm & GPU Environment")
print("=" * 55)

result = subprocess.run(
    ["rocm-smi", "--showproductname", "--showmeminfo", "vram"],
    capture_output=True, text=True,
)
print(result.stdout if result.returncode == 0 else "rocm-smi not found — skipping")

print(f"Python version   : {sys.version.split()[0]}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA (HIP) avail : {torch.cuda.is_available()}")
print(f"ROCm/HIP version : {getattr(torch.version, 'hip', 'N/A')}")
print(f"GPU count        : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  |  VRAM: {props.total_memory / 1024**3:.1f} GB")

total_vram = sum(
    torch.cuda.get_device_properties(i).total_memory
    for i in range(torch.cuda.device_count())
) / 1024**3
print(f"\nTotal VRAM       : {total_vram:.1f} GB")
print(f"BF16 supported   : {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

if total_vram >= 24:
    print("✅ Sufficient VRAM for π₀.₅ (expert-only, ~24 GB)")
else:
    print("⚠️  < 24 GB VRAM — enable train_expert_only + gradient_checkpointing")

major, minor = sys.version_info[:2]
if (major, minor) >= (3, 12):
    print("✅ Python 3.12+ satisfied")
else:
    print(f"⚠️  Python {major}.{minor} — LeRobot requires ≥ 3.12")

ROCm & GPU Environment


============================ ROCm System Management Interface ============================
================================== Memory Usage (Bytes) ==================================
GPU[0]		: VRAM Total Memory (B): 51522830336
GPU[0]		: VRAM Total Used Memory (B): 28012544
====================================== Product Info ======================================
GPU[0]		: Card Series: 		AMD Radeon Graphics
GPU[0]		: Card Model: 		0x744b
GPU[0]		: Card Vendor: 		Advanced Micro Devices, Inc. [AMD/ATI]
GPU[0]		: Card SKU: 		D7070910
GPU[0]		: Subsystem ID: 	0x0e0c
GPU[0]		: Device Rev: 		0x00
GPU[0]		: Node ID: 		2
GPU[0]		: GUID: 		6853
GPU[0]		: GFX Version: 		gfx1100
================================== End of ROCm SMI Log ===================================

Python version   : 3.12.3
PyTorch version  : 2.9.1+gitff65f5b
CUDA (HIP) avail : True
ROCm/HIP version : 7.2.53211-e1a6bc5663
GPU count        : 1
  GPU 0: AMD Radeon Graphics  |  VRAM: 48.0 GB

Total VRAM  

## 1. ROCm Environment Variables

Set these before importing torch or lerobot. They improve performance and compatibility on RDNA3/CDNA GPUs.

In [2]:
import os, subprocess

os.environ["TORCH_USE_HIP_DSA"] = "1"
os.environ["FLASH_ATTENTION_TRITON_AMD_ENABLE"] = "TRUE"
os.environ["PYTORCH_TUNABLEOP_ENABLED"] = "1"
os.environ["ROCR_VISIBLE_DEVICES"] = "0"
# os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"  # uncomment if ops fail on gfx1100

subprocess.run(["sudo", "sh", "-c", "echo 0 > /proc/sys/kernel/numa_balancing"], capture_output=True)

for k in ["TORCH_USE_HIP_DSA", "FLASH_ATTENTION_TRITON_AMD_ENABLE",
          "PYTORCH_TUNABLEOP_ENABLED", "ROCR_VISIBLE_DEVICES"]:
    print(f"  {k} = {os.environ[k]}")
print("✅ ROCm env set")

  TORCH_USE_HIP_DSA = 1
  FLASH_ATTENTION_TRITON_AMD_ENABLE = TRUE
  PYTORCH_TUNABLEOP_ENABLED = 1
  ROCR_VISIBLE_DEVICES = 0
✅ ROCm env set


## 2. Install LeRobot + π₀.₅ Dependencies

Installs `lerobot[pi]` directly from PyPI — no git clone needed.  
PyPI is reachable from this ROCm image; HuggingFace Hub and GitHub are not.

> **Note:** `ffmpeg` is optional — only needed for video dataset decoding. Inference works without it.

In [3]:
import subprocess, sys

def run(cmd):
    print(f"$ {' '.join(cmd)}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f"STDERR:\n{r.stderr[-1000:]}")
        return False
    print(r.stdout[-300:] or "(ok)")
    return True

ok = run([
    sys.executable, "-m", "pip", "install", "-q",
    "--trusted-host", "pypi.org",
    "--trusted-host", "files.pythonhosted.org",
    "lerobot[pi]",
])
print("✅ lerobot[pi] installed" if ok else "❌ install failed — see STDERR above")

# ffmpeg is optional (video decoding); inference works without it
if subprocess.run(["which", "ffmpeg"], capture_output=True).returncode != 0:
    run(["sudo", "apt-get", "install", "-y", "-q", "ffmpeg"]) or print("⚠️  ffmpeg unavailable (inference unaffected)")
else:
    print("✅ ffmpeg present")

$ /opt/venv/bin/python3.12 -m pip install -q --trusted-host pypi.org --trusted-host files.pythonhosted.org lerobot[pi]
(ok)
✅ lerobot[pi] installed
$ sudo apt-get install -y -q ffmpeg
STDERR:
E: Unable to locate package ffmpeg

⚠️  ffmpeg unavailable (inference unaffected)


## 3. Verify LeRobot Installation

In [4]:
import importlib, torch

for mod in ["lerobot", "torch", "transformers", "diffusers", "huggingface_hub"]:
    try:
        m = importlib.import_module(mod)
        print(f"✅ {mod} {getattr(m, '__version__', '?')}")
    except Exception as e:
        print(f"❌ {mod}: {e}")

try:
    from lerobot.policies.pi05.modeling_pi05 import PI05Policy
    print("✅ PI05Policy importable")
except ImportError as e:
    print(f"❌ PI05Policy: {e}")
    print("   Fix: pip install 'lerobot[pi]'")

print(f"\nCUDA (HIP) : {torch.cuda.is_available()}")
print(f"BF16       : {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}")

✅ lerobot 0.5.1
✅ torch 2.9.1+gitff65f5b
✅ transformers 5.3.0
✅ diffusers 0.35.2
✅ huggingface_hub 1.20.1
✅ PI05Policy importable

CUDA (HIP) : True
BF16       : True


## 4. Download π₀.₅ Model Weights

| Checkpoint | Description |
|---|---|
| `lerobot/pi05_base` | General base model |
| `lerobot/pi05_libero` | Finetuned on Libero benchmark (97.5% avg) |

This cell auto-detects HF connectivity, tries the `hf-mirror.com` mirror as fallback, and prints manual transfer instructions if both are unreachable.

In [5]:
import os, subprocess

MODEL_REPO = "lerobot/pi05_base"   # or "lerobot/pi05_libero"

# Weights land flat in this directory (config.json, *.safetensors, etc.)
WEIGHTS_DIR = os.path.join(os.path.expanduser("~"), "pi05_weights", MODEL_REPO.replace("/", "--"))
os.makedirs(WEIGHTS_DIR, exist_ok=True)

def _reachable(url):
    return subprocess.run(["curl", "-s", "--max-time", "5", url], capture_output=True).returncode == 0

if os.path.isfile(os.path.join(WEIGHTS_DIR, "config.json")):
    print(f"✅ Weights already present at {WEIGHTS_DIR}")
else:
    hf_ok     = _reachable("https://huggingface.co")
    mirror_ok = _reachable("https://hf-mirror.com")

    if hf_ok or mirror_ok:
        endpoint = "https://hf-mirror.com" if (mirror_ok and not hf_ok) else "https://huggingface.co"
        print(f"Downloading {MODEL_REPO} via {endpoint} ...")
        env = {**os.environ, "HF_ENDPOINT": endpoint}
        subprocess.run(
            ["hf", "download", MODEL_REPO, "--local-dir", WEIGHTS_DIR],
            env=env, check=True,
        )
        print("✅ Download complete.")
    else:
        print("⚠️  No HF connectivity. Transfer weights manually:")
        print()
        print("  On a machine WITH internet access:")
        print(f"    pip install huggingface_hub")
        print(f"    hf download {MODEL_REPO} --local-dir ~/pi05_weights/{MODEL_REPO.replace('/', '--')}")
        print(f"    scp -r ~/pi05_weights root@<cloud-ip>:~/pi05_weights")
        print()
        print(f"  Then re-run this cell.")

print(f"\nWeights dir: {WEIGHTS_DIR}")

✅ Weights already present at /root/pi05_weights/lerobot--pi05_base

Weights dir: /root/pi05_weights/lerobot--pi05_base


## 5. Load π₀.₅ Policy for Inference

Load the pretrained policy from the local cache using `device=cuda` and `dtype=bfloat16` for ROCm.

In [6]:
import os, torch
from lerobot.policies.pi05.modeling_pi05 import PI05Policy

MODEL_REPO  = "lerobot/pi05_base"
WEIGHTS_DIR = os.path.join(os.path.expanduser("~"), "pi05_weights", MODEL_REPO.replace("/", "--"))
DEVICE      = "cuda"
DTYPE       = torch.bfloat16

os.environ["HF_DATASETS_OFFLINE"]  = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

if not os.path.isfile(os.path.join(WEIGHTS_DIR, "config.json")):
    raise FileNotFoundError(
        f"No config.json in {WEIGHTS_DIR}\n"
        "Run Cell 4 (download-model) first, or transfer weights manually."
    )

print(f"Loading from {WEIGHTS_DIR}")
print(f"Device: {DEVICE}  |  dtype: {DTYPE}")
print(f"VRAM before: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

policy = PI05Policy.from_pretrained(
    WEIGHTS_DIR,
    local_files_only=True,
    device=DEVICE,
    dtype=DTYPE,
)
policy.eval()

vram_gb = torch.cuda.memory_allocated(0) / 1e9
n_params = sum(p.numel() for p in policy.parameters()) / 1e9
print(f"\n✅ Policy loaded.")
print(f"VRAM after  : {vram_gb:.2f} GB")
print(f"Parameters  : {n_params:.2f} B")

Loading from /root/pi05_weights/lerobot--pi05_base
Device: cuda  |  dtype: torch.bfloat16
VRAM before: 0.00 GB
The PI05 model is a direct port of the OpenPI implementation. 
This implementation follows the original OpenPI structure for compatibility. 
Original implementation: https://github.com/Physical-Intelligence/openpi


Loading model from: /root/pi05_weights/lerobot--pi05_base
✓ Loaded state dict from model.safetensors
Remapped 812 state dict keys
All keys loaded successfully!

✅ Policy loaded.
VRAM after  : 16.59 GB
Parameters  : 4.14 B


## 6. Run Inference (Dummy Observation)

π₀.₅ expects:
- **Images** from one or more cameras (RGB, `[B, C, H, W]`, float in `[0, 1]`)
- **Robot state** (joint positions + gripper) as a float tensor
- **Language instruction** as a string

Synthetic observation below verifies the forward pass on ROCm.

In [ ]:
import subprocess, sys, os, torch

# sentencepiece required for PaliGemma/Gemma slow tokenizer
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--trusted-host", "pypi.org",
     "--trusted-host", "files.pythonhosted.org",
     "sentencepiece"],
    check=True,
)

from transformers import AutoTokenizer, GemmaTokenizer

WEIGHTS_DIR = os.path.join(os.path.expanduser("~"), "pi05_weights", "lerobot--pi05_base")
DEVICE = "cuda"
DTYPE  = torch.bfloat16

# ── 1. Load tokenizer ─────────────────────────────────────────────────────────
print("Loading tokenizer ...")
try:
    tokenizer = AutoTokenizer.from_pretrained(
        WEIGHTS_DIR, local_files_only=True, use_fast=False
    )
except Exception:
    tokenizer = GemmaTokenizer.from_pretrained(
        WEIGHTS_DIR, local_files_only=True
    )
print(f"✅ Tokenizer: {type(tokenizer).__name__}  vocab={tokenizer.vocab_size}")

# ── 2. Build observation ──────────────────────────────────────────────────────
TASK = "Pick up the red cube and place it on the blue plate."

B = 1
H, W = 224, 224

img_base        = torch.rand(B, 3, H, W, device=DEVICE, dtype=DTYPE)
img_left_wrist  = torch.rand(B, 3, H, W, device=DEVICE, dtype=DTYPE)
img_right_wrist = torch.rand(B, 3, H, W, device=DEVICE, dtype=DTYPE)
state           = torch.zeros(B, 8, device=DEVICE, dtype=DTYPE)

enc = tokenizer(
    [TASK],
    return_tensors="pt",
    padding="max_length",
    max_length=48,
    truncation=True,
)
tokens = enc["input_ids"].to(DEVICE)              # Long  [B, seq_len]
attn   = enc["attention_mask"].bool().to(DEVICE)  # bool  [B, seq_len]  ← must be bool

batch = {
    "observation.images.base_0_rgb":        img_base,
    "observation.images.left_wrist_0_rgb":  img_left_wrist,
    "observation.images.right_wrist_0_rgb": img_right_wrist,
    "observation.state":                    state,
    "observation.language.tokens":          tokens,
    "observation.language.attention_mask":  attn,
}

print(f"Task   : '{TASK}'")
print(f"Tokens : {tokens.shape}  dtype={tokens.dtype}")
print(f"Attn   : {attn.shape}   dtype={attn.dtype}")

# ── 3. Run inference ──────────────────────────────────────────────────────────
print("\nRunning forward pass ...")
with torch.inference_mode():
    actions = policy.predict_action_chunk(batch)

print(f"\n✅ Inference complete!")
print(f"Action chunk shape : {actions.shape}")
print(f"Action sample      : {actions[0, 0].cpu().float().numpy()}")


## 7. Finetune π₀.₅ on a Custom Dataset

| Mode | Flag | VRAM | What trains |
|---|---|---|---|
| **Expert only** | `--policy.train_expert_only=true` | ~24 GB | Action expert + projections only |
| **Full finetune** | `--policy.train_expert_only=false` | ~40 GB | All params including VLM |

In [8]:
import os

DATASET_REPO = "your_hf_username/your_dataset"  # ← replace with your dataset
OUTPUT_DIR   = os.path.join(os.path.expanduser("~"), "pi05_finetune")
WEIGHTS_DIR  = os.path.join(os.path.expanduser("~"), "pi05_weights", "lerobot--pi05_base")
STEPS        = 3000
BATCH_SIZE   = 32
TRAIN_EXPERT = "true"   # "false" for full finetune (~40 GB VRAM)

cmd = [
    "lerobot-train",
    f"--dataset.repo_id={DATASET_REPO}",
    "--policy.type=pi05",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=pi05_rocm_finetune",
    f"--policy.pretrained_path={WEIGHTS_DIR}",
    "--policy.compile_model=true",
    "--policy.gradient_checkpointing=true",
    "--policy.dtype=bfloat16",
    "--policy.device=cuda",
    "--policy.freeze_vision_encoder=false",
    f"--policy.train_expert_only={TRAIN_EXPERT}",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--wandb.enable=false",
]

print("Training command:\n")
print("  " + " \\\n  ".join(cmd))
print("\nSet run_training=True in the next cell to launch.")

Training command:

  lerobot-train \
  --dataset.repo_id=your_hf_username/your_dataset \
  --policy.type=pi05 \
  --output_dir=/root/pi05_finetune \
  --job_name=pi05_rocm_finetune \
  --policy.pretrained_path=/root/pi05_weights/lerobot--pi05_base \
  --policy.compile_model=true \
  --policy.gradient_checkpointing=true \
  --policy.dtype=bfloat16 \
  --policy.device=cuda \
  --policy.freeze_vision_encoder=false \
  --policy.train_expert_only=true \
  --steps=3000 \
  --batch_size=32 \
  --wandb.enable=false

Set run_training=True in the next cell to launch.


In [9]:
import subprocess, os

run_training = False  # ← set True to launch

if run_training:
    env = {
        **os.environ,
        "TORCH_USE_HIP_DSA": "1",
        "FLASH_ATTENTION_TRITON_AMD_ENABLE": "TRUE",
        "PYTORCH_TUNABLEOP_ENABLED": "1",
        "ROCR_VISIBLE_DEVICES": "0",
    }
    print("Launching training ...")
    proc = subprocess.Popen(
        cmd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    for i, line in enumerate(proc.stdout):
        print(line, end="")
        if i >= 200:
            print("... (truncated — training continues in background)")
            break
else:
    print("Skipped — set run_training = True to launch.")

Skipped — set run_training = True to launch.
